# HELIOS Hybrid Graph Miner

This Colab notebook mines HELIOS documents and optional external literature to surface novel research connections, contradictions, and candidate branch paths.

## Setup

In [ ]:
!pip -q install pandas requests networkx scikit-learn numpy tqdm

import sys
print(f'Python {sys.version.split()[0]} setup ready')

## Mount Google Drive

## Configure Run

In [ ]:
CONFIG = {
    'repo_source_mode': 'github_raw',
    'repo_base_url': 'https://raw.githubusercontent.com/myrqyry/HELIOS-3D/main',
    'external_queries': [
        'hopfion spintronics',
        'compensated ferrimagnet spin pumping',
        'altermagnet skyrmion dynamics',
        'synthetic antiferromagnet skyrmion readout',
        'SOT domain wall logic planar',
        'van der Waals ferromagnet room temperature skyrmion',
    ],
    'max_external_papers': 15,
    'enable_live_search': True,
    'enable_contradiction_mining': True,
    'enable_github_writeback': False,
    'github_repo': 'myrqyry/HELIOS-3D',
    'drive_output_root': '/content/drive/MyDrive/helios-research',
}

CONFIG

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime
import json

drive.mount("/content/drive")

OUTPUT_ROOT = Path(CONFIG['drive_output_root'])
RUN_ID = datetime.utcnow().strftime('%Y%m%d-%H%M%S')
RUN_ROOT = OUTPUT_ROOT / RUN_ID
RAW_ROOT = RUN_ROOT / 'raw'
REPORT_ROOT = RUN_ROOT / 'reports'

for directory in (RAW_ROOT, REPORT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

print({
    'run_root': str(RUN_ROOT),
    'raw_root': str(RAW_ROOT),
    'report_root': str(REPORT_ROOT),
})


def save_text_artifact(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')


def save_json_artifact(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')

## Load HELIOS Docs

In [ ]:
import pandas as pd
import requests
from google.colab import files

DEFAULT_HELIOS_FILES = [
    'README.md',
    'docs/CLAIMS_MATRIX.md',
    'docs/ALTERNATIVE_MATERIALS_AND_METHODS.md',
    'docs/LITERATURE_REVIEW.md',
    'docs/OPEN_QUESTIONS.md',
    'docs/CANDIDATE_MATERIALS_AND_MECHANISMS.md',
    'docs/CORE_ARCHITECTURE.md',
]


def normalize_markdown(text: str) -> str:
    return '\n'.join(line.rstrip() for line in text.replace('\r\n', '\n').split('\n')).strip()


def fetch_github_text(base_url: str, relative_path: str) -> str:
    url = f"{base_url.rstrip('/')}/{relative_path.lstrip('/')}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return normalize_markdown(response.text)


def load_uploaded_documents() -> list[dict]:
    uploaded = files.upload()
    return [
        {
            'path': name,
            'text': normalize_markdown(content.decode('utf-8')),
            'source': 'helios',
        }
        for name, content in uploaded.items()
    ]


def load_helios_documents(config: dict) -> list[dict]:
    documents = []

    if config['repo_source_mode'] == 'upload':
        documents = load_uploaded_documents()
        for document in documents:
            save_text_artifact(RAW_ROOT / document['path'].replace('/', '__'), document['text'])
        return documents

    if config['repo_source_mode'] != 'github_raw':
        raise ValueError('repo_source_mode must be github_raw or upload')

    for relative_path in DEFAULT_HELIOS_FILES:
        text = fetch_github_text(config['repo_base_url'], relative_path)
        record = {
            'path': relative_path,
            'text': text,
            'source': 'helios',
        }
        documents.append(record)
        save_text_artifact(RAW_ROOT / relative_path.replace('/', '__'), text)

    return documents

In [ ]:
helios_docs = load_helios_documents(CONFIG)
helios_docs_df = pd.DataFrame([{'path': doc['path'], 'characters': len(doc['text'])} for doc in helios_docs])
save_json_artifact(RAW_ROOT / 'helios_docs_manifest.json', helios_docs_df.to_dict(orient='records'))
helios_docs_df

## Fetch External Literature

In [ ]:
from urllib.parse import quote_plus
import xml.etree.ElementTree as ET


def openalex_abstract_from_index(item: dict) -> str:
    inverted = item.get('abstract_inverted_index') or {}
    if not inverted:
        return item.get('abstract', '') or ''
    positions = []
    for token, indexes in inverted.items():
        for index in indexes:
            positions.append((index, token))
    return ' '.join(token for _, token in sorted(positions))


def fetch_arxiv_papers(query: str, max_results: int) -> list[dict]:
    url = (
        'https://export.arxiv.org/api/query?'
        f'search_query=all:{quote_plus(query)}&start=0&max_results={max_results}'
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    namespace = {'atom': 'http://www.w3.org/2005/Atom'}
    root = ET.fromstring(response.text)
    papers = []
    for entry in root.findall('atom:entry', namespace):
        papers.append(
            {
                'source': 'arxiv',
                'title': entry.findtext('atom:title', default='', namespaces=namespace).strip(),
                'abstract': entry.findtext('atom:summary', default='', namespaces=namespace).strip(),
                'url': entry.findtext('atom:id', default='', namespaces=namespace).strip(),
                'year': entry.findtext('atom:published', default='', namespaces=namespace)[:4],
                'identifier': entry.findtext('atom:id', default='', namespaces=namespace).strip(),
            }
        )
    return papers


def fetch_openalex_works(query: str, max_results: int) -> list[dict]:
    url = f"https://api.openalex.org/works?search={quote_plus(query)}&per-page={max_results}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    results = response.json().get('results', [])
    return [
        {
            'source': 'openalex',
            'title': item.get('display_name', ''),
            'abstract': openalex_abstract_from_index(item),
            'url': item.get('id', ''),
            'year': item.get('publication_year', ''),
            'identifier': item.get('id', ''),
        }
        for item in results
    ]


def fetch_crossref_works(query: str, max_results: int) -> list[dict]:
    url = f"https://api.crossref.org/works?query={quote_plus(query)}&rows={max_results}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    items = response.json().get('message', {}).get('items', [])
    return [
        {
            'source': 'crossref',
            'title': (item.get('title') or [''])[0],
            'abstract': item.get('abstract', '') or '',
            'url': item.get('URL', ''),
            'year': (item.get('published-print', {}).get('date-parts', [['']])[0][0]),
            'identifier': item.get('DOI', ''),
        }
        for item in items
    ]


def fetch_external_papers(config: dict) -> list[dict]:
    if not config['enable_live_search']:
        return []

    papers = []
    per_source = max(1, config['max_external_papers'] // 3)
    for query in config['external_queries']:
        papers.extend(fetch_arxiv_papers(query, per_source))
        papers.extend(fetch_openalex_works(query, per_source))
        papers.extend(fetch_crossref_works(query, per_source))

    deduped = {}
    for paper in papers:
        key = paper['identifier'] or paper['url'] or f"{paper['source']}::{paper['title']}"
        deduped[key] = paper

    records = list(deduped.values())[: config['max_external_papers']]
    save_json_artifact(RAW_ROOT / 'external_papers.json', records)
    return records

In [ ]:
external_papers = fetch_external_papers(CONFIG)
external_papers_df = pd.DataFrame(external_papers)
if not external_papers_df.empty:
    external_papers_df.to_csv(RAW_ROOT / 'external_papers.csv', index=False)
external_papers_df.head(10)

## Extract Claims and Build Graph

In [ ]:
import re
import networkx as nx

MATERIALS = [
    'Fe3GaTe2', 'Fe_3GaTe_2', 'Fe3GeTe2', 'GdFeCo', 'CrI3',
    'Ir/Fe/Co/Pt', 'Ir/Co/Pt', 'Pt/Co/Ta', 'W/CoFeB/MgO',
    'altermagnet', 'RuO2',
]
CARRIERS = ['skyrmion', 'hopfion', 'domain wall', 'skyrmionium', 'bimeron']
READOUTS = ['Hall', 'TMR', 'AHE', 'THE', 'microwave', 'spin pumping', 'ODMR', 'MOKE']
FABRICATION = ['ALD', 'DISH', 'TPP', 'sputter', 'etch', 'transfer', 'stacking', 'exfoliation']
STABILIZERS = ['DMI', 'moire', 'super-moire', 'twist', 'magnetic field', 'exchange bias']
TRANSPORTS = ['SOT', 'STT', 'spin Hall', 'Brownian', 'thermal', 'current-driven']
EVIDENCE_TAGS = ['[DEMONSTRATED]', '[INFERRED]', '[PROPOSED]', '[SPECULATIVE]']
NEGATION_KEYWORDS = [
    'unstable', 'not stable', 'fails at room temperature', 'high bias required',
    'degrades', 'not reproducible', 'limited to', 'remains challenging',
    'no experimental', 'theoretical only', 'speculative', 'has not been',
    'cannot be', 'difficult to', 'poor',
]


def extract_claim_units(documents: list[dict]) -> list[dict]:
    claims = []
    for doc in documents:
        for line in doc['text'].split('\n'):
            line = line.strip()
            if len(line) < 20:
                continue
            if line.startswith('#') or line.startswith('|') or line.startswith('- '):
                continue
            evidence_tag = None
            for tag in EVIDENCE_TAGS:
                if tag in line:
                    evidence_tag = tag
                    break
            claims.append({
                'path': doc['path'],
                'text': line,
                'source': 'helios',
                'evidence_tag': evidence_tag,
            })
    return claims


def extract_entities(text: str) -> dict:
    text_lower = text.lower()
    return {
        'materials': [item for item in MATERIALS if item.lower() in text_lower],
        'carriers': [item for item in CARRIERS if item.lower() in text_lower],
        'readouts': [item for item in READOUTS if item.lower() in text_lower],
        'fabrication': [item for item in FABRICATION if item.lower() in text_lower],
        'stabilizers': [item for item in STABILIZERS if item.lower() in text_lower],
        'transports': [item for item in TRANSPORTS if item.lower() in text_lower],
        'evidence_tags': [tag for tag in EVIDENCE_TAGS if tag in text],
    }


def detect_contradictions(paper_text: str) -> list[str]:
    paper_lower = paper_text.lower()
    found = []
    for keyword in NEGATION_KEYWORDS:
        if keyword.lower() in paper_lower:
            found.append(keyword)
    return found


def build_research_graph(claim_units: list[dict], papers: list[dict]) -> nx.MultiDiGraph:
    graph = nx.MultiDiGraph()

    for index, claim in enumerate(claim_units):
        claim_id = f'claim:{index}'
        graph.add_node(claim_id, kind='claim', path=claim['path'],
                       text=claim['text'], evidence_tag=claim.get('evidence_tag'))
        entities = extract_entities(claim['text'])
        for label, values in entities.items():
            for value in values:
                entity_id = f'entity:{value}'
                graph.add_node(entity_id, kind=label, label=value)
                graph.add_edge(claim_id, entity_id, relation='mentions')

        # Add evidence-tag-based edges
        tag = claim.get('evidence_tag')
        if tag == '[SPECULATIVE]':
            tag_node = 'entity:SPECULATIVE'
            graph.add_node(tag_node, kind='evidence_level', label='SPECULATIVE')
            graph.add_edge(claim_id, tag_node, relation='tagged_as')
        elif tag == '[DEMONSTRATED]':
            tag_node = 'entity:DEMONSTRATED'
            graph.add_node(tag_node, kind='evidence_level', label='DEMONSTRATED')
            graph.add_edge(claim_id, tag_node, relation='tagged_as')

    for index, paper in enumerate(papers):
        paper_id = f'paper:{index}'
        combined_text = f"{paper.get('title', '')} {paper.get('abstract', '')}"
        graph.add_node(paper_id, kind='paper', title=paper.get('title', ''),
                       abstract=paper.get('abstract', ''))
        entities = extract_entities(combined_text)
        for label, values in entities.items():
            for value in values:
                entity_id = f'entity:{value}'
                graph.add_node(entity_id, kind=label, label=value)
                graph.add_edge(paper_id, entity_id, relation='mentions')

        # Tag paper with contradictions detected
        neg_hits = detect_contradictions(combined_text)
        if neg_hits:
            graph.add_node(paper_id, contradiction_keywords=neg_hits)
            neg_node = 'entity:contradiction'
            graph.add_node(neg_node, kind='flag', label='contradiction')
            graph.add_edge(paper_id, neg_node, relation='contains_contradiction')

    return graph

In [ ]:
claim_units = extract_claim_units(helios_docs)
graph = build_research_graph(claim_units, external_papers)

graph_nodes = [{'node': node, **attrs} for node, attrs in graph.nodes(data=True)]
graph_edges = [
    {'source': source, 'target': target, **attrs}
    for source, target, attrs in graph.edges(data=True)
]

save_json_artifact(REPORT_ROOT / 'claim_units.json', claim_units)
save_json_artifact(REPORT_ROOT / 'graph_nodes.json', graph_nodes)
save_json_artifact(REPORT_ROOT / 'graph_edges.json', graph_edges)
pd.DataFrame(graph_nodes).to_csv(REPORT_ROOT / 'graph_nodes.csv', index=False)
pd.DataFrame(graph_edges).to_csv(REPORT_ROOT / 'graph_edges.csv', index=False)

print({
    'claims': len(claim_units),
    'graph_nodes': graph.number_of_nodes(),
    'graph_edges': graph.number_of_edges(),
})

## Rank Candidate Connections and Write Reports

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def classify_evidence_relationship(claim_text: str, paper_text: str,
                                    evidence_tag: str = None) -> str:
    claim_lower = claim_text.lower()
    paper_lower = paper_text.lower()

    # Check for contradictions first
    for keyword in NEGATION_KEYWORDS:
        if keyword.lower() in paper_lower and keyword.lower() not in claim_lower:
            return 'possible_contradiction'

    # Evidence-tag-aware classification
    if evidence_tag == '[SPECULATIVE]':
        has_direct = (
            'demonstrated' in paper_lower or 'measured' in paper_lower or
            'observed' in paper_lower or 'experimental' in paper_lower
        )
        if not has_direct:
            return 'demote_weak_support'
        return 'promote_to_inferred'

    if evidence_tag == '[PROPOSED]':
        has_experimental = (
            'experimental' in paper_lower or 'demonstrated' in paper_lower or
            'fabricated' in paper_lower
        )
        if has_experimental:
            return 'promote_to_demonstrated'
        return 'hold_needs_evidence'

    if evidence_tag == '[INFERRED]':
        has_confirmation = (
            'confirm' in paper_lower or 'consistent' in paper_lower or
            'supports' in paper_lower
        )
        if has_confirmation:
            return 'promote_to_demonstrated'
        return 'hold_partial'

    if evidence_tag == '[DEMONSTRATED]':
        return 'direct_support'

    if 'zero-field' in claim_lower and 'zero-field' not in paper_lower and 'reduced bias' in paper_lower:
        return 'partial_support'

    return 'direct_support'


def rank_candidate_connections(claim_units: list[dict], papers: list[dict]) -> list[dict]:
    if not claim_units or not papers:
        return []

    claim_texts = [claim['text'] for claim in claim_units]
    paper_texts = [f"{paper.get('title', '')} {paper.get('abstract', '')}" for paper in papers]
    matrix = TfidfVectorizer(stop_words='english').fit_transform(claim_texts + paper_texts)
    claim_matrix = matrix[: len(claim_texts)]
    paper_matrix = matrix[len(claim_texts):]
    scores = cosine_similarity(claim_matrix, paper_matrix)

    ranked = []
    for claim_index, claim in enumerate(claim_units):
        best_paper_index = scores[claim_index].argmax()
        paper = papers[int(best_paper_index)]
        paper_text = paper_texts[int(best_paper_index)]
        relation = classify_evidence_relationship(
            claim['text'], paper_text, claim.get('evidence_tag')
        )
        # Detect contradictions in paper
        contradictions = detect_contradictions(paper_text)
        ranked.append(
            {
                'claim_path': claim['path'],
                'claim_text': claim['text'],
                'evidence_tag': claim.get('evidence_tag'),
                'paper_title': paper.get('title', ''),
                'paper_url': paper.get('url', ''),
                'score': float(scores[claim_index][best_paper_index]),
                'relationship': relation,
                'contradictions_found': contradictions,
            }
        )

    return sorted(ranked, key=lambda item: item['score'], reverse=True)[:10]


def generate_claims_matrix_suggestions(ranked: list[dict]) -> list[dict]:
    suggestions = []
    for item in ranked:
        rel = item['relationship']
        if rel == 'promote_to_inferred' and item['evidence_tag'] == '[SPECULATIVE]':
            suggestions.append({
                'claim_text': item['claim_text'][:120],
                'current_tag': '[SPECULATIVE]',
                'suggested_tag': '[INFERRED]',
                'reason': f"External paper contains experimental evidence",
                'source': item['paper_url'],
            })
        elif rel == 'promote_to_demonstrated' and item['evidence_tag'] in ('[PROPOSED]', '[INFERRED]'):
            suggestions.append({
                'claim_text': item['claim_text'][:120],
                'current_tag': item['evidence_tag'],
                'suggested_tag': '[DEMONSTRATED]',
                'reason': f"External paper confirms with experimental data",
                'source': item['paper_url'],
            })
        elif rel == 'demote_weak_support':
            suggestions.append({
                'claim_text': item['claim_text'][:120],
                'current_tag': item['evidence_tag'] or '[SPECULATIVE]',
                'suggested_tag': 'DEMOTE',
                'reason': f"No experimental support found in external paper",
                'source': item['paper_url'],
            })
        elif item['contradictions_found']:
            suggestions.append({
                'claim_text': item['claim_text'][:120],
                'current_tag': item['evidence_tag'] or 'unknown',
                'suggested_tag': 'HIGH_RISK',
                'reason': f"Contradiction keywords: {item['contradictions_found']}",
                'source': item['paper_url'],
            })
    return suggestions


def create_github_issue_body(ranked: list[dict],
                              claims_suggestions: list[dict],
                              contradiction_flags: list[dict]) -> str:
    lines = [
        '# Automated Research Review',
        '',
        f'**Connections ranked:** {len(ranked)}',
        f'**Claims Matrix suggestions:** {len(claims_suggestions)}',
        f'**Contradiction flags:** {len(contradiction_flags)}',
        '',
        '## Top Candidate Connections',
        '',
    ]
    for item in ranked:
        tag = item.get('evidence_tag', 'none')
        lines.append(
            f"- `{item['relationship']}` [{tag}] `{item['claim_path']}` "
            f"| {item['paper_title']} | score={item['score']:.3f}"
        )
    lines.extend([
        '',
        '## Claims Matrix Update Suggestions',
        '',
    ])
    if claims_suggestions:
        for s in claims_suggestions:
            lines.append(f"- **{s['current_tag']}** -> **{s['suggested_tag']}**: {s['reason']}")
            lines.append(f"  - Source: {s['source']}")
    else:
        lines.append('No promotion/demotion suggestions at this time.')
    lines.extend([
        '',
        '## Contradiction Flags',
        '',
    ])
    if contradiction_flags:
        for cf in contradiction_flags:
            lines.append(f"- ! {cf['claim_path']}: {cf['paper_title']}")
            lines.append(f"  - Keywords: {cf['contradictions_found']}")
    else:
        lines.append('No contradictions detected in this run.')
    return '\n'.join(lines)


def github_create_issue(config: dict, body: str, timestamp: str) -> str:
    import os
    token = os.environ.get('GITHUB_TOKEN', '')
    if not token:
        return 'GITHUB_TOKEN not set. Skipping GitHub issue creation.'
    import requests
    url = f"https://api.github.com/repos/{config['github_repo']}/issues"
    headers = {
        'Authorization': f'token {token}',
        'Accept': 'application/vnd.github.v3+json',
    }
    payload = {
        'title': f'Automated Research Review {timestamp}',
        'body': body,
        'labels': ['automated-research', 'graph-miner'],
    }
    resp = requests.post(url, json=payload, headers=headers, timeout=30)
    if resp.status_code == 201:
        return f"Issue created: {resp.json().get('html_url', '')}"
    return f"GitHub API error: {resp.status_code} {resp.text[:200]}"


def write_markdown_report(ranked_connections: list[dict],
                           claims_suggestions: list[dict],
                           contradiction_flags: list[dict]) -> str:
    lines = [
        '# HELIOS Hybrid Graph Miner Report',
        '',
        '## Top 10 Candidate Connections',
        '',
    ]
    for item in ranked_connections:
        tag = item.get('evidence_tag', 'none')
        lines.append(
            f"- `{item['relationship']}` [{tag}] `{item['claim_path']}` "
            f"| {item['paper_title']} | score={item['score']:.3f}"
        )
    lines.extend([
        '',
        '## Promote / Hold / Demote',
        '',
    ])
    if claims_suggestions:
        for s in claims_suggestions:
            lines.append(f"- **{s['current_tag']} -> {s['suggested_tag']}**: {s['reason']}")
    else:
        lines.append('- No automatic promotion/demotion recommendations at this time.')
    lines.extend([
        '',
        '## Contradiction Mining Results',
        '',
    ])
    if contradiction_flags:
        lines.append(f'**{len(contradiction_flags)} contradiction flags found:**')
        for cf in contradiction_flags:
            lines.append(f"- {cf['claim_path']}: {cf['paper_title']}")
    else:
        lines.append('- No contradictions detected.')
    lines.extend([
        '',
        '## How to Use This Report',
        '',
        '- **Promote**: high-score connections tied to the planar-first baseline.',
        '- **Hold**: partial-support branches that need matching conditions.',
        '- **Demote**: attractive connections with no fabrication or readout path.',
        '- **HIGH_RISK**: claims contradicted by external literature findings.',
    ])
    return '\n'.join(lines)

In [ ]:
ranked_connections = rank_candidate_connections(claim_units, external_papers)

# Contradiction mining
contradiction_flags = []
if CONFIG['enable_contradiction_mining']:
    for item in ranked_connections:
        if item['contradictions_found']:
            item['relationship'] = 'CONTRADICTION'
            contradiction_flags.append(item)
else:
    for item in ranked_connections:
        if item['relationship'] in ('partial_support', 'possible_contradiction'):
            item['relationship'] = 'direct_support'

# Generate Claims Matrix update suggestions
claims_suggestions = generate_claims_matrix_suggestions(ranked_connections)

# Build report
report_markdown = write_markdown_report(ranked_connections, claims_suggestions, contradiction_flags)
ranked_df = pd.DataFrame(ranked_connections)

# Save artifacts
save_json_artifact(REPORT_ROOT / 'ranked_connections.json', ranked_connections)
save_json_artifact(REPORT_ROOT / 'claims_suggestions.json', claims_suggestions)
save_json_artifact(REPORT_ROOT / 'contradiction_flags.json', contradiction_flags)
ranked_df.to_csv(REPORT_ROOT / 'ranked_connections.csv', index=False)
if claims_suggestions:
    pd.DataFrame(claims_suggestions).to_csv(REPORT_ROOT / 'claims_suggestions.csv', index=False)
if contradiction_flags:
    pd.DataFrame(contradiction_flags).to_csv(REPORT_ROOT / 'contradiction_flags.csv', index=False)
save_text_artifact(REPORT_ROOT / 'report.md', report_markdown)

# GitHub write-back
if CONFIG.get('enable_github_writeback'):
    from datetime import datetime
    timestamp = datetime.utcnow().strftime('%Y-%m-%d')
    issue_body = create_github_issue_body(ranked_connections, claims_suggestions, contradiction_flags)
    issue_result = github_create_issue(CONFIG, issue_body, timestamp)
    save_text_artifact(REPORT_ROOT / 'github_issue_result.txt', issue_result)
    print(f'GitHub writeback: {issue_result}')

print(report_markdown)
print(f'\nContradiction flags: {len(contradiction_flags)}')
print(f'Claims suggestions: {len(claims_suggestions)}')
ranked_df.head(10)